## YouTube download setup

This downloader is intended for publicly available videos and does not use browser sign-in or cookies.

Before running the downloader:

1. Make sure the playlist is publicly accessible.
2. **Kaggle:** turn on **Notebook options > Internet**, then run the Deno setup cell. It installs Deno inside the Kaggle notebook environment.
3. **Local Windows:** the Deno setup cell detects the installed Deno automatically.
4. Do not export or share cookie files; this notebook does not need them.

Private, age-restricted, members-only, or region-restricted videos may still require authentication and cannot be downloaded by this cookie-free version.

In [1]:
%pip install -U "yt-dlp[default]" opencv-python

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: C:\Users\nishi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
import os
import shutil
import subprocess
import urllib.request


def refresh_deno_path():
    """Add common Windows, Linux, and Kaggle Deno locations to PATH."""
    candidates = [
        # Windows
        os.path.expandvars(r"%USERPROFILE%\.deno\bin"),
        os.path.expandvars(r"%LOCALAPPDATA%\Microsoft\WinGet\Links"),
        os.path.expandvars(
            r"%LOCALAPPDATA%\Microsoft\WinGet\Packages\DenoLand.Deno_Microsoft.Winget.Source_8wekyb3d8bbwe"
        ),
        r"C:\Program Files\Deno",
        # Linux and Kaggle
        os.path.expanduser("~/.deno/bin"),
        "/kaggle/working/.deno/bin",
    ]
    current_path = os.environ.get("PATH", "").split(os.pathsep)
    for path in candidates:
        if os.path.isdir(path) and path not in current_path:
            current_path.append(path)
    os.environ["PATH"] = os.pathsep.join(current_path)


def install_deno_on_kaggle():
    """Install Deno in Kaggle when it is not already available."""
    if not os.path.isdir("/kaggle") or shutil.which("deno"):
        return

    print("Kaggle detected; installing Deno in the notebook environment...")
    install_script = "/tmp/install_deno.sh"
    urllib.request.urlretrieve("https://deno.land/install.sh", install_script)
    subprocess.run(["bash", install_script], check=True)
    refresh_deno_path()


refresh_deno_path()
install_deno_on_kaggle()
refresh_deno_path()
deno_executable = shutil.which("deno")

if deno_executable is None:
    print("Deno is unavailable.")
    print("On Kaggle, turn on Notebook options > Internet and run this cell again.")
    print("On Windows, install Deno and restart VS Code.")
else:
    print("Deno executable:", deno_executable)
    version_result = subprocess.run(
        [deno_executable, "--version"],
        check=False,
        capture_output=True,
        text=True,
    )
    if version_result.returncode == 0:
        print(version_result.stdout.strip())
    else:
        print("Deno was found, but version check failed:")
        print(version_result.stderr.strip())

Deno executable: C:\Users\nishi\AppData\Local\Microsoft\WinGet\Packages\DenoLand.Deno_Microsoft.Winget.Source_8wekyb3d8bbwe\deno.EXE
deno 2.9.5 (stable, release, x86_64-pc-windows-msvc)
v8 15.0.245.2-rusty
typescript 6.0.3


In [1]:
import glob
import os
import re
import shutil
import subprocess
import time

import cv2
import yt_dlp


# ============================================================
# SETTINGS
# ============================================================

PLAYLIST_URL = "https://youtube.com/playlist?list=PLb9PqCCW7Q08&si=3usr95BKg_gWFDZy"
VIDEO_DIR = "temp_video"
IMAGE_DIR = "images"
START_VIDEO = 1
INTERVAL = 10


os.makedirs(VIDEO_DIR, exist_ok=True)
os.makedirs(IMAGE_DIR, exist_ok=True)

existing_files = glob.glob(os.path.join(IMAGE_DIR, "train*.jpg"))
numbers = []
for file in existing_files:
    match = re.match(r"train(\d+)\.jpg", os.path.basename(file))
    if match:
        numbers.append(int(match.group(1)))
image_counter = max(numbers, default=0) + 1
failed_videos = []
ffmpeg_executable = shutil.which("ffmpeg")


def numeric_image_sort(path):
    match = re.search(r"train(\d+)\.jpg$", os.path.basename(path))
    return int(match.group(1)) if match else 0


print("Next image will be:", f"train{image_counter}.jpg")
print("Downloading public playlist videos without browser cookies")
if ffmpeg_executable:
    print("FFmpeg found; using fast single-pass frame extraction")
else:
    print("FFmpeg was not found; using OpenCV frame extraction")

playlist_opts = {
    "skip_download": True,
    "ignoreerrors": True,
    "js_runtimes": {"deno": {}},
}

with yt_dlp.YoutubeDL(playlist_opts) as ydl:
    playlist = ydl.extract_info(PLAYLIST_URL, download=False)

videos = [video for video in playlist.get("entries", []) if video]
videos = videos[START_VIDEO - 1:]
print("Videos to process:", len(videos))

for video_index, video in enumerate(videos, start=START_VIDEO):
    video_started_at = time.perf_counter()
    print("\n" + "=" * 60)
    print(f"Processing playlist video {video_index}")
    print("=" * 60)

    video_url = video.get("webpage_url") or video.get("original_url") or video.get("url")
    if not video_url:
        print("No usable URL was found; skipping this video.")
        failed_videos.append((video_index, "No usable URL"))
        print(f"Video {video_index} elapsed time: {time.perf_counter() - video_started_at:.2f} seconds")
        continue
    print("URL:", video_url)

    output_template = os.path.join(VIDEO_DIR, "current.%(ext)s")
    for old_file in glob.glob(os.path.join(VIDEO_DIR, "current.*")):
        try:
            os.remove(old_file)
        except OSError:
            pass

    download_opts = {
        "format": "bv*[height<=1080]+ba/b[height<=1080]",
        "merge_output_format": "mp4",
        "outtmpl": output_template,
        "ignoreerrors": True,
        "js_runtimes": {"deno": {}},
    }

    try:
        with yt_dlp.YoutubeDL(download_opts) as ydl:
            result = ydl.download([video_url])
    except Exception as error:
        print("DOWNLOAD FAILED; continuing with the next video:")
        print(error)
        failed_videos.append((video_index, str(error)))
        for temporary_file in glob.glob(os.path.join(VIDEO_DIR, "current.*")):
            try:
                os.remove(temporary_file)
            except OSError:
                pass
        print(f"Video {video_index} elapsed time: {time.perf_counter() - video_started_at:.2f} seconds")
        continue

    downloaded = [
        file for file in glob.glob(os.path.join(VIDEO_DIR, "current.*"))
        if file.lower().endswith((".mp4", ".mkv", ".webm"))
    ]
    video_path = downloaded[0] if downloaded else None
    if video_path is None or result not in (None, 0):
        print("Video unavailable or download failed; skipping it.")
        failed_videos.append((video_index, "Unavailable or download failed"))
        for temporary_file in glob.glob(os.path.join(VIDEO_DIR, "current.*")):
            try:
                os.remove(temporary_file)
            except OSError:
                pass
        print(f"Video {video_index} elapsed time: {time.perf_counter() - video_started_at:.2f} seconds")
        continue

    print("Downloaded:", video_path)
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print("Could not open video.")
        failed_videos.append((video_index, "Could not open video"))
        cap.release()
        os.remove(video_path)
        print(f"Video {video_index} elapsed time: {time.perf_counter() - video_started_at:.2f} seconds")
        continue

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = total_frames / fps if fps > 0 else 0

    if fps <= 0:
        print("Invalid FPS.")
        failed_videos.append((video_index, "Invalid FPS"))
        cap.release()
        os.remove(video_path)
        print(f"Video {video_index} elapsed time: {time.perf_counter() - video_started_at:.2f} seconds")
        continue

    print("Resolution:", width, "x", height)
    print("FPS:", fps)
    print("Duration:", duration, "seconds")

    existing_image_paths = set(glob.glob(os.path.join(IMAGE_DIR, "train*.jpg")))
    extracted_with_ffmpeg = False
    extracted_frames = []

    if ffmpeg_executable:
        ffmpeg_command = [
            ffmpeg_executable,
            "-hide_banner",
            "-loglevel",
            "error",
            "-i",
            video_path,
            "-vf",
            f"fps=1/{INTERVAL}",
            "-q:v",
            "2",
            "-start_number",
            str(image_counter),
            os.path.join(IMAGE_DIR, "train%d.jpg"),
        ]
        ffmpeg_result = subprocess.run(
            ffmpeg_command,
            check=False,
            capture_output=True,
            text=True,
        )
        extracted_frames = sorted(
            set(glob.glob(os.path.join(IMAGE_DIR, "train*.jpg")))
            - existing_image_paths,
            key=numeric_image_sort,
        )
        extracted_with_ffmpeg = ffmpeg_result.returncode == 0 and bool(extracted_frames)
        if not extracted_with_ffmpeg:
            print("FFmpeg extraction failed; falling back to OpenCV.")
            if ffmpeg_result.stderr.strip():
                print(ffmpeg_result.stderr.strip())
            for frame_path in extracted_frames:
                try:
                    os.remove(frame_path)
                except OSError:
                    pass
            extracted_frames = []

    if extracted_with_ffmpeg:
        for frame_number, frame_path in enumerate(extracted_frames):
            print(f"Saved {os.path.basename(frame_path)} | Video: {video_index} | Time: {frame_number * INTERVAL}s")
            image_counter += 1
    else:
        timestamp = 0
        while timestamp < duration:
            cap.set(cv2.CAP_PROP_POS_MSEC, timestamp * 1000)
            success, frame = cap.read()
            if success:
                filename = f"train{image_counter}.jpg"
                output_path = os.path.join(IMAGE_DIR, filename)
                if cv2.imwrite(output_path, frame):
                    print(f"Saved {filename} | Video: {video_index} | Time: {timestamp}s")
                    image_counter += 1
                else:
                    print("Failed to save frame at:", timestamp, "seconds")
            else:
                print("Could not read frame at:", timestamp, "seconds")
            timestamp += INTERVAL

    cap.release()
    for temporary_file in glob.glob(os.path.join(VIDEO_DIR, "current.*")):
        try:
            os.remove(temporary_file)
            print("Deleted temporary video file:", temporary_file)
        except OSError as error:
            print("Could not delete temporary file:", error)

    print(f"Video {video_index} elapsed time: {time.perf_counter() - video_started_at:.2f} seconds")

print("\n" + "=" * 60)
print("DONE")
print("Next image number would be:", f"train{image_counter}.jpg")
if failed_videos:
    print("Skipped videos:", len(failed_videos))
    for video_index, reason in failed_videos:
        print(f"  Video {video_index}: {reason}")
else:
    print("No videos were skipped.")
print("=" * 60)

Next image will be: train1.jpg
FFmpeg found; using fast single-pass frame extraction
[youtube:tab] Extracting URL: https://youtube.com/playlist?list=PLb9PqCCW7Q08&si=3usr95BKg_gWFDZy
[youtube:tab] PLb9PqCCW7Q08: Downloading webpage


[youtube:tab] PLb9PqCCW7Q08: Redownloading playlist API JSON with unavailable videos
[download] Downloading playlist: World Cup Matches - Tactical CAM
[youtube:tab] PLb9PqCCW7Q08 page 1: Downloading API JSON
[youtube:tab] Playlist World Cup Matches - Tactical CAM: Downloading 35 items of 35
[download] Downloading item 1 of 35
[youtube] Extracting URL: https://www.youtube.com/watch?v=t_nEnsBQ988
[youtube] t_nEnsBQ988: Downloading webpage


[youtube] t_nEnsBQ988: Downloading visionos player API JSON
[youtube] t_nEnsBQ988: Downloading m3u8 information
[download] Downloading item 2 of 35
[youtube] Extracting URL: https://www.youtube.com/watch?v=sL_RzJ4sixs
[youtube] sL_RzJ4sixs: Downloading webpage


[youtube] sL_RzJ4sixs: Downloading visionos player API JSON
[youtube] sL_RzJ4sixs: Downloading m3u8 information
[download] Downloading item 3 of 35
[youtube] Extracting URL: https://www.youtube.com/watch?v=KaZ09PNp3Lw
[youtube] KaZ09PNp3Lw: Downloading webpage


[youtube] KaZ09PNp3Lw: Downloading visionos player API JSON
[youtube] KaZ09PNp3Lw: Downloading m3u8 information
[download] Downloading item 4 of 35
[youtube] Extracting URL: https://www.youtube.com/watch?v=2xrrRLlEW38
[youtube] 2xrrRLlEW38: Downloading webpage


...

[youtube] I0k5dSPMZVs: Downloading visionos player API JSON
[youtube] I0k5dSPMZVs: Downloading m3u8 information
[info] I0k5dSPMZVs: Downloading 1 format(s): 136+251
[download] Destination: temp_video\current.f136.mp4
[download] 100% of  932.84MiB in 00:03:24 at 4.55MiB/s      
[download] Destination: temp_video\current.f251.webm
[download] 100% of    3.72MiB in 00:00:00 at 5.75MiB/s   
[Merger] Merging formats into "temp_video\current.mp4"
Deleting original file temp_video\current.f136.mp4 (pass -k to keep)
Deleting original file temp_video\current.f251.webm (pass -k to keep)
Downloaded: temp_video\current.mp4
Resolution: 1280 x 720
FPS: 29.97002997002997
Duration: 8622.5139 seconds
Saved train19315.jpg | Video: 29 | Time: 0s
Saved train19316.jpg | Video: 29 | Time: 10s
Saved train19317.jpg | Video: 29 | Time: 20s
Saved train19318.jpg | Video: 29 | Time: 30s
Saved train19319.jpg | Video: 29 | Time: 40s
Saved train19320.jpg | Video: 29 | Time: 50s
Saved train19321.jpg | Video: 29 | Time

[youtube] 1dwMA34pQLA: Downloading visionos player API JSON
[youtube] 1dwMA34pQLA: Downloading m3u8 information
[info] 1dwMA34pQLA: Downloading 1 format(s): 136+251
[download] Destination: temp_video\current.f136.mp4
[download] 100% of    1.19GiB in 00:04:34 at 4.45MiB/s      
[download] Destination: temp_video\current.f251.webm
[download] 100% of    4.03MiB in 00:00:00 at 6.02MiB/s   
[Merger] Merging formats into "temp_video\current.mp4"
Deleting original file temp_video\current.f136.mp4 (pass -k to keep)
Deleting original file temp_video\current.f251.webm (pass -k to keep)
Downloaded: temp_video\current.mp4
Resolution: 1280 x 720
FPS: 29.97002997002997
Duration: 9337.194533333333 seconds
Saved train20177.jpg | Video: 30 | Time: 0s
Saved train20178.jpg | Video: 30 | Time: 10s
Saved train20179.jpg | Video: 30 | Time: 20s
Saved train20180.jpg | Video: 30 | Time: 30s
Saved train20181.jpg | Video: 30 | Time: 40s
Saved train20182.jpg | Video: 30 | Time: 50s
Saved train20183.jpg | Video: 3

[youtube] dsqoZKUAoF0: Downloading visionos player API JSON
[youtube] dsqoZKUAoF0: Downloading m3u8 information
[info] dsqoZKUAoF0: Downloading 1 format(s): 136+251
[download] Destination: temp_video\current.f136.mp4
[download] 100% of  882.85MiB in 00:01:30 at 9.77MiB/s      
[download] Destination: temp_video\current.f251.webm
[download] 100% of    2.70MiB in 00:00:01 at 2.42MiB/s   
[Merger] Merging formats into "temp_video\current.mp4"
Deleting original file temp_video\current.f136.mp4 (pass -k to keep)
Deleting original file temp_video\current.f251.webm (pass -k to keep)
Downloaded: temp_video\current.mp4
Resolution: 1280 x 720
FPS: 29.97002997002997
Duration: 6257.484566666667 seconds
Saved train21111.jpg | Video: 31 | Time: 0s
Saved train21112.jpg | Video: 31 | Time: 10s
Saved train21113.jpg | Video: 31 | Time: 20s
Saved train21114.jpg | Video: 31 | Time: 30s
Saved train21115.jpg | Video: 31 | Time: 40s
Saved train21116.jpg | Video: 31 | Time: 50s
Saved train21117.jpg | Video: 3

[youtube] -AAjsqD1-Tk: Downloading visionos player API JSON
[youtube] -AAjsqD1-Tk: Downloading m3u8 information
[info] -AAjsqD1-Tk: Downloading 1 format(s): 137+251
[download] Destination: temp_video\current.f137.mp4
[download] 100% of    1.69GiB in 00:02:53 at 9.94MiB/s      
[download] Destination: temp_video\current.f251.webm
[download] 100% of    2.59MiB in 00:00:00 at 5.83MiB/s   
[Merger] Merging formats into "temp_video\current.mp4"
Deleting original file temp_video\current.f251.webm (pass -k to keep)
Deleting original file temp_video\current.f137.mp4 (pass -k to keep)
Downloaded: temp_video\current.mp4
Resolution: 1920 x 1080
FPS: 29.97002997002997
Duration: 6001.5288666666665 seconds
Saved train21737.jpg | Video: 32 | Time: 0s
Saved train21738.jpg | Video: 32 | Time: 10s
Saved train21739.jpg | Video: 32 | Time: 20s
Saved train21740.jpg | Video: 32 | Time: 30s
Saved train21741.jpg | Video: 32 | Time: 40s
Saved train21742.jpg | Video: 32 | Time: 50s
Saved train21743.jpg | Video:

[youtube] XGBPLjH-2ko: Downloading visionos player API JSON
[youtube] XGBPLjH-2ko: Downloading m3u8 information
[info] XGBPLjH-2ko: Downloading 1 format(s): 136+251
[download] Destination: temp_video\current.f136.mp4
[download] 100% of 1020.03MiB in 00:02:01 at 8.41MiB/s      
[download] Destination: temp_video\current.f251.webm
[download] 100% of    3.56MiB in 00:00:00 at 3.58MiB/s   
[Merger] Merging formats into "temp_video\current.mp4"
Deleting original file temp_video\current.f136.mp4 (pass -k to keep)
Deleting original file temp_video\current.f251.webm (pass -k to keep)
Downloaded: temp_video\current.mp4
Resolution: 1280 x 720
FPS: 29.97002997002997
Duration: 8236.1279 seconds
Saved train22337.jpg | Video: 33 | Time: 0s
Saved train22338.jpg | Video: 33 | Time: 10s
Saved train22339.jpg | Video: 33 | Time: 20s
Saved train22340.jpg | Video: 33 | Time: 30s
Saved train22341.jpg | Video: 33 | Time: 40s
Saved train22342.jpg | Video: 33 | Time: 50s
Saved train22343.jpg | Video: 33 | Time

In [2]:
import glob
import os
import re
import shutil
import subprocess
import time

import cv2
import yt_dlp


# ============================================================
# SETTINGS
# ============================================================

PLAYLIST_URL = "https://youtube.com/playlist?list=PL29faCH1zKViiz6tMwTwvZy43Fgow5RIA&si=CpXI_9pqYmULM4hE"
VIDEO_DIR = "temp_video"
IMAGE_DIR = "images"
START_VIDEO = 1
INTERVAL = 10


os.makedirs(VIDEO_DIR, exist_ok=True)
os.makedirs(IMAGE_DIR, exist_ok=True)

existing_files = glob.glob(os.path.join(IMAGE_DIR, "train*.jpg"))
numbers = []
for file in existing_files:
    match = re.match(r"train(\d+)\.jpg", os.path.basename(file))
    if match:
        numbers.append(int(match.group(1)))
image_counter = max(numbers, default=0) + 1
failed_videos = []
ffmpeg_executable = shutil.which("ffmpeg")


def numeric_image_sort(path):
    match = re.search(r"train(\d+)\.jpg$", os.path.basename(path))
    return int(match.group(1)) if match else 0


print("Next image will be:", f"train{image_counter}.jpg")
print("Downloading public playlist videos without browser cookies")
if ffmpeg_executable:
    print("FFmpeg found; using fast single-pass frame extraction")
else:
    print("FFmpeg was not found; using OpenCV frame extraction")

playlist_opts = {
    "skip_download": True,
    "ignoreerrors": True,
    "js_runtimes": {"deno": {}},
}

with yt_dlp.YoutubeDL(playlist_opts) as ydl:
    playlist = ydl.extract_info(PLAYLIST_URL, download=False)

videos = [video for video in playlist.get("entries", []) if video]
videos = videos[START_VIDEO - 1:]
print("Videos to process:", len(videos))

for video_index, video in enumerate(videos, start=START_VIDEO):
    video_started_at = time.perf_counter()
    print("\n" + "=" * 60)
    print(f"Processing playlist video {video_index}")
    print("=" * 60)

    video_url = video.get("webpage_url") or video.get("original_url") or video.get("url")
    if not video_url:
        print("No usable URL was found; skipping this video.")
        failed_videos.append((video_index, "No usable URL"))
        print(f"Video {video_index} elapsed time: {time.perf_counter() - video_started_at:.2f} seconds")
        continue
    print("URL:", video_url)

    output_template = os.path.join(VIDEO_DIR, "current.%(ext)s")
    for old_file in glob.glob(os.path.join(VIDEO_DIR, "current.*")):
        try:
            os.remove(old_file)
        except OSError:
            pass

    download_opts = {
        "format": "bv*[height<=1080]+ba/b[height<=1080]",
        "merge_output_format": "mp4",
        "outtmpl": output_template,
        "ignoreerrors": True,
        "js_runtimes": {"deno": {}},
    }

    try:
        with yt_dlp.YoutubeDL(download_opts) as ydl:
            result = ydl.download([video_url])
    except Exception as error:
        print("DOWNLOAD FAILED; continuing with the next video:")
        print(error)
        failed_videos.append((video_index, str(error)))
        for temporary_file in glob.glob(os.path.join(VIDEO_DIR, "current.*")):
            try:
                os.remove(temporary_file)
            except OSError:
                pass
        print(f"Video {video_index} elapsed time: {time.perf_counter() - video_started_at:.2f} seconds")
        continue

    downloaded = [
        file for file in glob.glob(os.path.join(VIDEO_DIR, "current.*"))
        if file.lower().endswith((".mp4", ".mkv", ".webm"))
    ]
    video_path = downloaded[0] if downloaded else None
    if video_path is None or result not in (None, 0):
        print("Video unavailable or download failed; skipping it.")
        failed_videos.append((video_index, "Unavailable or download failed"))
        for temporary_file in glob.glob(os.path.join(VIDEO_DIR, "current.*")):
            try:
                os.remove(temporary_file)
            except OSError:
                pass
        print(f"Video {video_index} elapsed time: {time.perf_counter() - video_started_at:.2f} seconds")
        continue

    print("Downloaded:", video_path)
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print("Could not open video.")
        failed_videos.append((video_index, "Could not open video"))
        cap.release()
        os.remove(video_path)
        print(f"Video {video_index} elapsed time: {time.perf_counter() - video_started_at:.2f} seconds")
        continue

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = total_frames / fps if fps > 0 else 0

    if fps <= 0:
        print("Invalid FPS.")
        failed_videos.append((video_index, "Invalid FPS"))
        cap.release()
        os.remove(video_path)
        print(f"Video {video_index} elapsed time: {time.perf_counter() - video_started_at:.2f} seconds")
        continue

    print("Resolution:", width, "x", height)
    print("FPS:", fps)
    print("Duration:", duration, "seconds")

    existing_image_paths = set(glob.glob(os.path.join(IMAGE_DIR, "train*.jpg")))
    extracted_with_ffmpeg = False
    extracted_frames = []

    if ffmpeg_executable:
        ffmpeg_command = [
            ffmpeg_executable,
            "-hide_banner",
            "-loglevel",
            "error",
            "-i",
            video_path,
            "-vf",
            f"fps=1/{INTERVAL}",
            "-q:v",
            "2",
            "-start_number",
            str(image_counter),
            os.path.join(IMAGE_DIR, "train%d.jpg"),
        ]
        ffmpeg_result = subprocess.run(
            ffmpeg_command,
            check=False,
            capture_output=True,
            text=True,
        )
        extracted_frames = sorted(
            set(glob.glob(os.path.join(IMAGE_DIR, "train*.jpg")))
            - existing_image_paths,
            key=numeric_image_sort,
        )
        extracted_with_ffmpeg = ffmpeg_result.returncode == 0 and bool(extracted_frames)
        if not extracted_with_ffmpeg:
            print("FFmpeg extraction failed; falling back to OpenCV.")
            if ffmpeg_result.stderr.strip():
                print(ffmpeg_result.stderr.strip())
            for frame_path in extracted_frames:
                try:
                    os.remove(frame_path)
                except OSError:
                    pass
            extracted_frames = []

    if extracted_with_ffmpeg:
        for frame_number, frame_path in enumerate(extracted_frames):
            print(f"Saved {os.path.basename(frame_path)} | Video: {video_index} | Time: {frame_number * INTERVAL}s")
            image_counter += 1
    else:
        timestamp = 0
        while timestamp < duration:
            cap.set(cv2.CAP_PROP_POS_MSEC, timestamp * 1000)
            success, frame = cap.read()
            if success:
                filename = f"train{image_counter}.jpg"
                output_path = os.path.join(IMAGE_DIR, filename)
                if cv2.imwrite(output_path, frame):
                    print(f"Saved {filename} | Video: {video_index} | Time: {timestamp}s")
                    image_counter += 1
                else:
                    print("Failed to save frame at:", timestamp, "seconds")
            else:
                print("Could not read frame at:", timestamp, "seconds")
            timestamp += INTERVAL

    cap.release()
    for temporary_file in glob.glob(os.path.join(VIDEO_DIR, "current.*")):
        try:
            os.remove(temporary_file)
            print("Deleted temporary video file:", temporary_file)
        except OSError as error:
            print("Could not delete temporary file:", error)

    print(f"Video {video_index} elapsed time: {time.perf_counter() - video_started_at:.2f} seconds")

print("\n" + "=" * 60)
print("DONE")
print("Next image number would be:", f"train{image_counter}.jpg")
if failed_videos:
    print("Skipped videos:", len(failed_videos))
    for video_index, reason in failed_videos:
        print(f"  Video {video_index}: {reason}")
else:
    print("No videos were skipped.")
print("=" * 60)

Next image will be: train23161.jpg
FFmpeg found; using fast single-pass frame extraction
[youtube:tab] Extracting URL: https://youtube.com/playlist?list=PL29faCH1zKViiz6tMwTwvZy43Fgow5RIA&si=CpXI_9pqYmULM4hE
[youtube:tab] PL29faCH1zKViiz6tMwTwvZy43Fgow5RIA: Downloading webpage


[youtube:tab] PL29faCH1zKViiz6tMwTwvZy43Fgow5RIA: Redownloading playlist API JSON with unavailable videos
[download] Downloading playlist: Football Matches Tactical Cam
[youtube:tab] PL29faCH1zKViiz6tMwTwvZy43Fgow5RIA page 1: Downloading API JSON
[youtube:tab] Playlist Football Matches Tactical Cam: Downloading 64 items of 64
[download] Downloading item 1 of 64
[youtube] Extracting URL: https://www.youtube.com/watch?v=KgQquW68E-Q
[youtube] KgQquW68E-Q: Downloading webpage


[youtube] KgQquW68E-Q: Downloading visionos player API JSON
[youtube] KgQquW68E-Q: Downloading m3u8 information
[download] Downloading item 2 of 64
[youtube] Extracting URL: https://www.youtube.com/watch?v=gzNLfgbxsLk
[youtube] gzNLfgbxsLk: Downloading webpage


[youtube] gzNLfgbxsLk: Downloading visionos player API JSON


ERROR: [youtube] gzNLfgbxsLk: Video unavailable


...

[youtube] a8txafr1VNI: Downloading visionos player API JSON
[youtube] a8txafr1VNI: Downloading m3u8 information
[info] a8txafr1VNI: Downloading 1 format(s): 398+251
[download] Destination: temp_video\current.f398.mp4
[download] 100% of  640.34MiB in 00:01:00 at 10.61MiB/s    
[download] Destination: temp_video\current.f251.webm
[download] 100% of   94.39MiB in 00:00:09 at 9.69MiB/s     
[Merger] Merging formats into "temp_video\current.mp4"
Deleting original file temp_video\current.f251.webm (pass -k to keep)
Deleting original file temp_video\current.f398.mp4 (pass -k to keep)
Downloaded: temp_video\current.mp4
Resolution: 1280 x 720
FPS: 30.0
Duration: 6763.566666666667 seconds
Saved train51587.jpg | Video: 54 | Time: 0s
Saved train51588.jpg | Video: 54 | Time: 10s
Saved train51589.jpg | Video: 54 | Time: 20s
Saved train51590.jpg | Video: 54 | Time: 30s
Saved train51591.jpg | Video: 54 | Time: 40s
Saved train51592.jpg | Video: 54 | Time: 50s
Saved train51593.jpg | Video: 54 | Time: 60

[youtube] 5gM-Fjgc2tk: Downloading visionos player API JSON
[youtube] 5gM-Fjgc2tk: Downloading m3u8 information
[info] 5gM-Fjgc2tk: Downloading 1 format(s): 137+251
[download] Destination: temp_video\current.f137.mp4
[download] 100% of    2.69GiB in 00:06:32 at 7.01MiB/s      
[download] Destination: temp_video\current.f251.webm
[download] 100% of  133.76MiB in 00:00:40 at 3.31MiB/s     
[Merger] Merging formats into "temp_video\current.mp4"
Deleting original file temp_video\current.f251.webm (pass -k to keep)
Deleting original file temp_video\current.f137.mp4 (pass -k to keep)
Downloaded: temp_video\current.mp4
Resolution: 1920 x 1080
FPS: 29.97002997002997
Duration: 8964.956 seconds
Saved train52263.jpg | Video: 55 | Time: 0s
Saved train52264.jpg | Video: 55 | Time: 10s
Saved train52265.jpg | Video: 55 | Time: 20s
Saved train52266.jpg | Video: 55 | Time: 30s
Saved train52267.jpg | Video: 55 | Time: 40s
Saved train52268.jpg | Video: 55 | Time: 50s
Saved train52269.jpg | Video: 55 | Ti

[youtube] vkyCLzUvv7c: Downloading initial data API JSON


[youtube] vkyCLzUvv7c: Downloading visionos player API JSON
[youtube] vkyCLzUvv7c: Downloading m3u8 information
[info] vkyCLzUvv7c: Downloading 1 format(s): 616+251
[hlsnative] Downloading m3u8 manifest
[hlsnative] Total fragments: 1907
[download] Destination: temp_video\current.f616.mp4
[download] 100% of    3.67GiB in 00:41:07 at 1.52MiB/s                        
[download] Destination: temp_video\current.f251.webm
[download] 100% of  150.70MiB in 00:01:00 at 2.49MiB/s     
[Merger] Merging formats into "temp_video\current.mp4"
Deleting original file temp_video\current.f616.mp4 (pass -k to keep)
Deleting original file temp_video\current.f251.webm (pass -k to keep)
Downloaded: temp_video\current.mp4
Resolution: 1920 x 1080
FPS: 30.0
Duration: 10612.2 seconds
Saved train53159.jpg | Video: 56 | Time: 0s
Saved train53160.jpg | Video: 56 | Time: 10s
Saved train53161.jpg | Video: 56 | Time: 20s
Saved train53162.jpg | Video: 56 | Time: 30s
Saved train53163.jpg | Video: 56 | Time: 40s
Saved 

[youtube] 6lAbqQIvQUo: Downloading visionos player API JSON
[youtube] 6lAbqQIvQUo: Downloading m3u8 information
[info] 6lAbqQIvQUo: Downloading 1 format(s): 398+251
[download] Destination: temp_video\current.f398.mp4
[download] 100% of  794.63MiB in 00:02:57 at 4.48MiB/s      
[download] Destination: temp_video\current.f251.webm
[download] 100% of  126.32MiB in 00:00:23 at 5.30MiB/s     
[Merger] Merging formats into "temp_video\current.mp4"
Deleting original file temp_video\current.f251.webm (pass -k to keep)
Deleting original file temp_video\current.f398.mp4 (pass -k to keep)
Downloaded: temp_video\current.mp4
Resolution: 1280 x 720
FPS: 29.97002997002997
Duration: 8558.816933333334 seconds
Saved train54220.jpg | Video: 57 | Time: 0s
Saved train54221.jpg | Video: 57 | Time: 10s
Saved train54222.jpg | Video: 57 | Time: 20s
Saved train54223.jpg | Video: 57 | Time: 30s
Saved train54224.jpg | Video: 57 | Time: 40s
Saved train54225.jpg | Video: 57 | Time: 50s
Saved train54226.jpg | Video:

In [3]:
import os
import glob
import shutil
import random
import re


# ============================================================
# SETTINGS
# ============================================================

IMAGE_DIR = "images"

TRAIN_DIR = "images/train"
TEST_DIR = "images/test"

TEST_RATIO = 0.20

RANDOM_SEED = 42


# ============================================================
# CREATE FOLDERS
# ============================================================

os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(TEST_DIR, exist_ok=True)


# ============================================================
# GET ALL IMAGES
# ============================================================

images = glob.glob(
    os.path.join(
        IMAGE_DIR,
        "train*.jpg"
    )
)


# ============================================================
# SORT BY ORIGINAL NUMBER
# ============================================================

def get_number(path):

    filename = os.path.basename(path)

    match = re.search(
        r"train(\d+)\.jpg",
        filename
    )

    if match:
        return int(match.group(1))

    return -1


images.sort(
    key=get_number
)


print(
    "Total images:",
    len(images)
)


# ============================================================
# SHUFFLE
# ============================================================

random.seed(RANDOM_SEED)

random.shuffle(images)


# ============================================================
# CALCULATE TEST SIZE
# ============================================================

test_size = int(
    len(images) * TEST_RATIO
)


test_images = images[:test_size]

train_images = images[test_size:]


print(
    "Training images:",
    len(train_images)
)

print(
    "Testing images:",
    len(test_images)
)


# ============================================================
# COPY TRAIN IMAGES
# ============================================================

for image_path in train_images:

    filename = os.path.basename(
        image_path
    )

    destination = os.path.join(
        TRAIN_DIR,
        filename
    )

    shutil.copy2(
        image_path,
        destination
    )


# ============================================================
# COPY + RENAME TEST IMAGES
# ============================================================

for i, image_path in enumerate(
    test_images,
    start=1
):

    new_filename = (
        f"test{i}.jpg"
    )

    destination = os.path.join(
        TEST_DIR,
        new_filename
    )

    shutil.copy2(
        image_path,
        destination
    )


# ============================================================
# DONE
# ============================================================

print("\nDONE")

print(
    "Train folder:",
    TRAIN_DIR
)

print(
    "Test folder:",
    TEST_DIR
)

Total images: 55075
Training images: 44060
Testing images: 11015

DONE
Train folder: images/train
Test folder: images/test
